# Text Search Demo — Visual Search, Target Acquisition, and Verification Analysis

This notebook analyzes real output from the heterogeneous text-search task. The central distinction is between **finding** the answer-relevant phrase and **verifying** information after it has been found.

The workflow uses `DataLoader`, `FixationAnalyzer`, and `ScanpathsAnalyzer` from `tobii-pytracker`. Word-level AOI scoring is performed against the `words` bounding boxes saved with each trial because the current public `BBoxAttentionAnalyzer` is specific to `image_bboxes`.

The analysis is descriptive/exploratory. It demonstrates a reproducible workflow rather than a confirmatory statistical model.

## Scientific context and experimental logic

This notebook follows the scientific framing in [Text Search Demo](../../../docs/basic_examples/text_search_demo.md). The task is a **goal-directed reading and information-retrieval paradigm**: the participant receives a practical question, searches a short text, and responds `YES`, `NO`, or `I DON'T KNOW`.

### Experimental structure

- 12 response-gated text trials;
- 6 `EARLY` trials with the critical phrase near the beginning of the text;
- 6 `LATE` trials with the critical phrase near the end;
- heterogeneous applied topics such as returns, delivery, accounts, refunds, warranties, invoices, pickup, and personal data;
- a known `critical_phrase` for each item provides an explicit answer-relevant AOI;
- every stimulus preserves `QUESTION`, a blank separator line, and `TEXT` in both the visible layout and word-bbox geometry.

Unlike the more tightly paired UX A/B Text Demo, this example intentionally contains stronger **item heterogeneity**. That makes it useful for demonstrating applied search analysis, but it also means that placement effects must be interpreted alongside topic and text-level variability.

### Process model

The documentation describes five conceptually distinct stages:

1. **Initial orientation** — where visual exploration begins.
2. **Search progression** — the sequence of fixations before the critical phrase.
3. **Target acquisition** — the first fixation on answer-relevant information.
4. **Verification** — continued or repeated inspection after acquisition.
5. **Decision outcome** — the final behavioral response.

The notebook preserves that separation rather than collapsing all gaze behavior into a single score.

| Documentation hypothesis | Operational measure in this notebook | Primary interpretation |
|---|---|---|
| H1 — Critical-phrase attention | `target_seen`, acquisition rate | Was the critical phrase visually reached? |
| H2 — Position effect | target TTFF by EARLY/LATE | Did placement alter access time? |
| H3 — Pre-target search | fixations before target | How much search preceded acquisition? |
| H4 — Verification behavior | target revisits, post-target fixations/dwell | Was information revisited or checked after discovery? |
| H5 — Behavioral association | accuracy by acquisition / verification profile | Is gaze behavior associated with correct responding? |

### Measurement caution

The notebook treats fixation as evidence of overt visual attention, not a direct readout of comprehension. Missing gaze is reported as missing eye-tracking evidence rather than coded as target avoidance. Because item topic and wording vary, category-level summaries are accompanied by item-level diagnostics before any substantive interpretation.

The demo documentation cites Rayner (1998) and Duchowski (2017) as methodological background for reading and eye-movement analysis.


## 1. Analysis parameters

All event-detection parameters are explicit so the notebook can be reproduced and audited. Fixation settings affect TTFF, pre-target fixation count, dwell, and revisit metrics, while scanpath summaries depend on the resulting event sequence.

Because this demo contains heterogeneous text items, parameter tuning should never be based on which setting produces the clearest EARLY/LATE difference. Use the sensitivity section to test robustness instead.


In [ ]:
LANGUAGE = "en"  # "en" or "pl"
SELECTED_SESSION = None
FIXATION_PARAMS = {"method":"dispersion", "dispersion_threshold":50.0, "min_duration":0.10}
RUN_FIXATION_SENSITIVITY = False
SENSITIVITY_DISPERSION_THRESHOLDS = [40.0, 50.0, 60.0]


## 2. Load a real session

The analysis operates only on collected experiment output. `DataLoader` is used to keep trial metadata, gaze samples, screenshots, and geometry synchronized. If no valid session exists, the notebook stops instead of substituting synthetic data.

For longitudinal or multi-participant work, replace the convenience 'latest session' selection with an explicit session list and preserve participant identifiers outside the raw data files.

> **tobii-pytracker support:** `CustomConfig` and `DataLoader` provide the canonical session-loading layer used throughout this notebook.


In [ ]:
from pathlib import Path
import ast
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tobii_pytracker.configs.custom_config import CustomConfig
from tobii_pytracker.analyze import (
    DataLoader,
    FixationAnalyzer,
    ScanpathsAnalyzer,
)


def find_demo_root() -> Path:
    """Locate tobii-pytracker-demo from common Jupyter launch locations."""
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for candidate in candidates:
        if (candidate / "examples").is_dir() and (candidate / "output").is_dir():
            return candidate
        nested = candidate / "tobii-pytracker-demo"
        if (nested / "examples").is_dir() and (nested / "output").is_dir():
            return nested
    raise FileNotFoundError("Could not locate the tobii-pytracker-demo repository root.")


def newest_subject(loader: DataLoader) -> str:
    subjects = loader.get_subjects()
    if not subjects:
        raise FileNotFoundError(f"No experiment sessions found under {loader.output_root}")
    def mtime(subject: str) -> float:
        return (loader.output_root / subject / "data.csv").stat().st_mtime
    return max(subjects, key=mtime)


def prepare_session(config_path: Path, subject: str | None = None):
    """Load one real session with DataLoader; never synthesize replacement data."""
    config = CustomConfig(str(config_path))
    loader = DataLoader(config=config, root=DEMO_ROOT)
    selected = subject or newest_subject(loader)
    raw = loader.get_subject_data(selected, flatten=False).reset_index(drop=True)
    raw.insert(0, "set_name", selected)
    raw["slide_index"] = np.arange(len(raw), dtype=int)
    flat = loader.get_subject_data(selected, flatten=True)
    return loader, selected, raw, flat


def safe_parse(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    text = "" if value is None else str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default


def point_in_centered_bbox(x: float, y: float, bbox: dict, margin: float = 2.0) -> bool:
    try:
        cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
    except (KeyError, TypeError, ValueError):
        return False
    return (cx - w/2 - margin <= x <= cx + w/2 + margin and
            cy - h/2 - margin <= y <= cy + h/2 + margin)


def normalize_token(value) -> str:
    return re.sub(r"[^0-9a-ząćęłńóśźż]+", "", str(value).casefold())


def trial_gaze_counts(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    counts = pd.Series(0, index=range(n_trials), dtype=int)
    if not flat.empty and {"slide_index", "avg_gaze_x"}.issubset(flat.columns):
        observed = flat.dropna(subset=["avg_gaze_x", "avg_gaze_y"]).groupby("slide_index").size()
        for idx, count in observed.items():
            if int(idx) in counts.index:
                counts.loc[int(idx)] = int(count)
    return counts


def trial_observed_duration(flat: pd.DataFrame, n_trials: int) -> pd.Series:
    duration = pd.Series(np.nan, index=range(n_trials), dtype=float)
    if not flat.empty and {"slide_index", "system_time"}.issubset(flat.columns):
        for idx, group in flat.dropna(subset=["system_time"]).groupby("slide_index"):
            if len(group) >= 2 and int(idx) in duration.index:
                duration.loc[int(idx)] = float(group["system_time"].max() - group["system_time"].min())
    return duration


def run_fixations(flat: pd.DataFrame, output_dir: Path, params: dict) -> pd.DataFrame:
    required = {"set_name", "slide_index", "avg_gaze_x", "avg_gaze_y", "system_time"}
    if flat.empty or not required.issubset(flat.columns):
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    clean = flat.dropna(subset=["avg_gaze_x","avg_gaze_y","system_time"]).copy()
    if clean.empty:
        return pd.DataFrame(columns=["set_name","slide_index","fix_start","fix_end","duration","x_mean","y_mean","dispersion"])
    analyzer = FixationAnalyzer(output_dir, **params)
    return analyzer.analyze(clean)


def run_scanpaths(fixations: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    if fixations.empty:
        return pd.DataFrame(columns=["set_name","slide_index","distance"])
    return ScanpathsAnalyzer(output_dir).analyze(fixations, per="slide")


def summarize_fixations(fixations: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if fixations.empty:
        return base.assign(fixation_count=0, total_fixation_duration=0.0, mean_fixation_duration=np.nan)
    agg = (fixations.groupby("slide_index")
           .agg(fixation_count=("duration","size"),
                total_fixation_duration=("duration","sum"),
                mean_fixation_duration=("duration","mean"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"fixation_count":0,"total_fixation_duration":0.0})


def summarize_scanpaths(scanpaths: pd.DataFrame, n_trials: int) -> pd.DataFrame:
    base = pd.DataFrame({"slide_index": range(n_trials)})
    if scanpaths.empty:
        return base.assign(scanpath_transition_count=0, scanpath_distance=0.0)
    agg = (scanpaths.groupby("slide_index")
           .agg(scanpath_transition_count=("distance","size"), scanpath_distance=("distance","sum"))
           .reset_index())
    return base.merge(agg, on="slide_index", how="left").fillna({"scanpath_transition_count":0,"scanpath_distance":0.0})

DEMO_ROOT = find_demo_root()
print(f"Demo repository: {DEMO_ROOT}")

In [ ]:
example_dir = DEMO_ROOT / "examples" / "text_search_demo"
config_candidates = sorted(example_dir.glob("config*.yaml"))
config_path = next(p for p in config_candidates if (p.stem.endswith("_pl")) == (LANGUAGE == "pl"))
dataset_name = "text_search.csv" if LANGUAGE == "en" else "text_search_pl.csv"
loader, SESSION, raw, flat = prepare_session(config_path, SELECTED_SESSION)
dataset_path = example_dir / "data" / dataset_name
stimuli = pd.read_csv(dataset_path)
lookup = stimuli.set_index("selected_text", drop=False)
analysis_dir = loader.output_root / SESSION / "analysis_text_search_demo_v2"
analysis_dir.mkdir(exist_ok=True)

for column in ["item_id","topic","condition","answer","critical_phrase","question","text_word_count"]:
    raw[column] = raw["input_data"].map(lookup[column])
raw["response"] = raw["user_classification"].astype(str).str.casefold()
raw["correct"] = raw["response"] == raw["answer"].astype(str).str.casefold()
raw["uncertain"] = raw["response"].isin(["none", "i don't know", "nie wiem"])
print(f"Session: {SESSION}; trials={len(raw)}; gaze samples={len(flat)}")

## 3. Reproducibility record and design integrity

This section fingerprints the analyzed session and stimulus table and checks the expected 12-trial structure. The goal is to establish **what was analyzed** before interpreting any eye-movement metric.

Because content topics vary across items, design verification also protects against accidental omission or duplication that could distort topic- or placement-level summaries.


In [ ]:
import hashlib
import platform
from importlib.metadata import PackageNotFoundError, version as package_version


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def installed_version(distribution: str) -> str:
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return "package-metadata-unavailable"


def provenance_table(session: str, data_csv: Path, dataset_path: Path, analysis_label: str) -> pd.DataFrame:
    record = {
        "analysis_label": analysis_label,
        "session": str(session),
        "data_csv_sha256": sha256_file(data_csv),
        "dataset_sha256": sha256_file(dataset_path),
        "python": platform.python_version(),
        "tobii_pytracker": installed_version("tobii-pytracker"),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    }
    return pd.DataFrame([record])


def save_json(path: Path, payload: dict):
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")

data_csv = loader.output_root / SESSION / "data.csv"
provenance = provenance_table(SESSION, data_csv, dataset_path, "text_search_demo_v2")
design_checks = pd.DataFrame([
    {"check": "12 recorded trials", "passed": len(raw) == 12, "observed": len(raw), "expected": 12},
    {"check": "EARLY/LATE balance", "passed": raw["condition"].value_counts().to_dict() == {"EARLY": 6, "LATE": 6}, "observed": str(raw["condition"].value_counts().to_dict()), "expected": "EARLY=6, LATE=6"},
    {"check": "YES/NO stimulus balance", "passed": stimuli["answer"].str.casefold().value_counts().to_dict() == {"yes": 6, "no": 6}, "observed": str(stimuli["answer"].str.casefold().value_counts().to_dict()), "expected": "yes=6, no=6"},
    {"check": "12 distinct topics", "passed": stimuli["topic"].nunique() == 12, "observed": int(stimuli["topic"].nunique()), "expected": 12},
    {"check": "metadata mapping complete", "passed": raw[["item_id","topic","condition","answer","critical_phrase","text_word_count"]].notna().all().all(), "observed": int(raw[["item_id","topic","condition","answer","critical_phrase","text_word_count"]].isna().sum().sum()), "expected": 0},
])
display(provenance); display(design_checks)
if not bool(design_checks["passed"].all()): warnings.warn("The selected session is incomplete or does not match the canonical demo design.")

## 4. Data-quality audit

A behavioral response and a usable gaze stream are separate properties. Trials with valid responses but missing gaze stay in the behavioral dataset and are explicitly marked unavailable for eye-tracking metrics.

This is especially important for H1 and H5: missing gaze must not be converted into `target_seen=False`, because that would confound recording failure with visual search behavior.


In [ ]:

gaze_counts = trial_gaze_counts(flat, len(raw)); observed_duration = trial_observed_duration(flat, len(raw))
quality = raw[["slide_index","item_id","topic","condition","response","correct","uncertain"]].copy()
quality["gaze_samples"] = quality["slide_index"].map(gaze_counts)
quality["usable_gaze"] = quality["gaze_samples"] > 0
quality["observed_gaze_duration_s"] = quality["slide_index"].map(observed_duration)
quality_overview = pd.DataFrame([
    {"metric":"recorded_trials","value":len(quality)},
    {"metric":"trials_with_usable_gaze","value":int(quality["usable_gaze"].sum())},
    {"metric":"trials_without_usable_gaze","value":int((~quality["usable_gaze"]).sum())},
    {"metric":"uncertain_responses","value":int(quality["uncertain"].sum())},
])
display(quality_overview); display(quality)
if (~quality["usable_gaze"]).any(): warnings.warn("Behavioral responses are retained; eye-movement metrics are unavailable on trials without gaze.")


## 5. Inspect one representative trial

Inspecting a real trial provides a geometry and timing sanity check before aggregation. The flattened gaze stream shows sample-level data, while the screenshot overlay helps verify that gaze aligns with the displayed question/text layout.

The stored blank line between `QUESTION` and `TEXT` should be visible both on screen and in the word-box geometry used for target scoring.

> **tobii-pytracker support:** Trial inspection uses `DataLoader.get_slide_data()` and the library-provided gaze plotting helper.


In [ ]:

if quality["usable_gaze"].any():
    representative = int(quality.loc[quality["usable_gaze"]].sort_values("gaze_samples",ascending=False).iloc[0]["slide_index"])
    slide = loader.get_slide_data(SESSION, representative, flatten=True)
    display(slide.head()); loader.plot_gaze(SESSION, representative, gradient=True, show=True)
else:
    representative = None
    print("No usable gaze is available for screenshot-level diagnostics.")


## 6. Fixation and scanpath extraction

The notebook uses the built-in `FixationAnalyzer` and `ScanpathsAnalyzer` so the example demonstrates the library's supported event-processing workflow. Fixations provide stable viewing events; scanpath transitions describe how visual search progresses spatially.

These measures are interpreted as components of search behavior, not as direct measurements of comprehension.

> **tobii-pytracker support:** This section delegates event extraction and scanpath summaries to `FixationAnalyzer` and `ScanpathsAnalyzer`.


In [ ]:
fixations = run_fixations(flat, analysis_dir, FIXATION_PARAMS)
scanpaths = run_scanpaths(fixations, analysis_dir)
fix_summary = summarize_fixations(fixations, len(raw)); scan_summary = summarize_scanpaths(scanpaths, len(raw))

## 7. Optional fixation-parameter sensitivity

Enable this section when evaluating how robust fixation-dependent conclusions are to reasonable dispersion-threshold changes.

In [ ]:
sensitivity = pd.DataFrame()
if RUN_FIXATION_SENSITIVITY:
    rows = []
    for threshold in SENSITIVITY_DISPERSION_THRESHOLDS:
        params = dict(FIXATION_PARAMS)
        params["dispersion_threshold"] = float(threshold)
        detected = run_fixations(flat, analysis_dir / "sensitivity", params)
        rows.append({
            "dispersion_threshold": float(threshold),
            "fixation_count": int(len(detected)),
            "mean_fixation_duration": float(detected["duration"].mean()) if not detected.empty else np.nan,
        })
    sensitivity = pd.DataFrame(rows)
    display(sensitivity)
else:
    print("Sensitivity analysis is disabled. Set RUN_FIXATION_SENSITIVITY=True to compare fixation thresholds.")

## 8. Critical-phrase AOI and verification metrics

The known `critical_phrase` provides a task-defined AOI for each item. Fixation centroids are scored against the word boxes that compose that phrase. The first hit defines target acquisition; later returns support descriptive measures of verification.

The notebook separates **pre-target search** from **post-target verification** because identical total fixation counts can arise from very different search strategies.

> **tobii-pytracker support:** The fixation stream is generated by `FixationAnalyzer`, while the phrase-level AOI match is experiment-specific logic built on the word geometry recorded in the output.


In [ ]:

def target_word_boxes(objects_bboxes, phrase: str):
    """Return word AOIs for the first contiguous match of the predefined target phrase."""
    objects = safe_parse(objects_bboxes, dict, {})
    words = objects.get("words", []) if isinstance(objects, dict) else []
    tokens = [normalize_token(w.get("word", "")) for w in words if isinstance(w, dict)]
    target = [normalize_token(token) for token in str(phrase).split()]
    target = [token for token in target if token]
    if not target:
        return []
    width = len(target)
    for start in range(0, len(tokens) - width + 1):
        if tokens[start:start + width] == target:
            return [words[i].get("bbox", {}) for i in range(start, start + width)]
    return []


def target_fixation_metrics(raw: pd.DataFrame, fixations: pd.DataFrame, gaze_flat: pd.DataFrame, phrase_column: str) -> pd.DataFrame:
    """Score fixation centroids against the target phrase without treating missing gaze as target absence."""
    rows = []
    for _, trial in raw.iterrows():
        slide = int(trial["slide_index"])
        boxes = target_word_boxes(trial.get("objects_bboxes"), trial[phrase_column])
        gaze = gaze_flat[gaze_flat["slide_index"] == slide].dropna(subset=["avg_gaze_x", "avg_gaze_y"]) if not gaze_flat.empty else pd.DataFrame()
        has_gaze = not gaze.empty
        f = fixations[fixations["slide_index"] == slide].sort_values("fix_start").copy() if not fixations.empty else pd.DataFrame()

        base = {"slide_index": slide, "target_bbox_found": bool(boxes), "target_bbox_count": int(len(boxes))}
        if not has_gaze:
            rows.append({**base, "target_seen": np.nan, "ttff_target_s": np.nan,
                         "fixations_before_target": np.nan, "pre_target_dwell_s": np.nan,
                         "target_fixation_count": np.nan, "target_dwell_s": np.nan,
                         "target_dwell_share": np.nan, "target_revisits": np.nan,
                         "post_target_fixation_count": np.nan, "post_target_fixation_duration_s": np.nan,
                         "post_target_elapsed_s": np.nan})
            continue
        if f.empty:
            rows.append({**base, "target_seen": False, "ttff_target_s": np.nan,
                         "fixations_before_target": 0, "pre_target_dwell_s": 0.0,
                         "target_fixation_count": 0, "target_dwell_s": 0.0,
                         "target_dwell_share": 0.0, "target_revisits": 0,
                         "post_target_fixation_count": 0, "post_target_fixation_duration_s": 0.0,
                         "post_target_elapsed_s": 0.0})
            continue

        hit = f.apply(lambda r: any(point_in_centered_bbox(r["x_mean"], r["y_mean"], b) for b in boxes), axis=1)
        f = f.assign(target_hit=hit.to_numpy())
        total_dwell = float(f["duration"].sum())
        trial_start = float(gaze["system_time"].min()) if "system_time" in gaze and gaze["system_time"].notna().any() else float(f["fix_start"].min())
        if bool(f["target_hit"].any()):
            first_pos = int(np.flatnonzero(f["target_hit"].to_numpy())[0])
            first = f.iloc[first_pos]
            target_fix = f[f["target_hit"]]
            sequence = f["target_hit"].astype(bool).tolist()
            entries = int(sequence[0]) + sum(sequence[i] and not sequence[i - 1] for i in range(1, len(sequence)))
            revisits = max(entries - 1, 0)
            after = f.iloc[first_pos + 1:]
            target_dwell = float(target_fix["duration"].sum())
            post_elapsed = max(0.0, float(f["fix_end"].max() - first["fix_end"]))
            rows.append({**base, "target_seen": True,
                         "ttff_target_s": max(0.0, float(first["fix_start"] - trial_start)),
                         "fixations_before_target": first_pos,
                         "pre_target_dwell_s": float(f.iloc[:first_pos]["duration"].sum()),
                         "target_fixation_count": int(len(target_fix)),
                         "target_dwell_s": target_dwell,
                         "target_dwell_share": target_dwell / total_dwell if total_dwell else np.nan,
                         "target_revisits": int(revisits),
                         "post_target_fixation_count": int(len(after)),
                         "post_target_fixation_duration_s": float(after["duration"].sum()),
                         "post_target_elapsed_s": post_elapsed})
        else:
            rows.append({**base, "target_seen": False, "ttff_target_s": np.nan,
                         "fixations_before_target": int(len(f)), "pre_target_dwell_s": total_dwell,
                         "target_fixation_count": 0, "target_dwell_s": 0.0,
                         "target_dwell_share": 0.0, "target_revisits": 0,
                         "post_target_fixation_count": 0, "post_target_fixation_duration_s": 0.0,
                         "post_target_elapsed_s": 0.0})
    return pd.DataFrame(rows)


def plot_target_aoi(loader, session, raw, fixations, slide_index: int, phrase_column: str):
    """Overlay the predefined target AOI and detected fixation centroids on the recorded screenshot."""
    from matplotlib.patches import Rectangle
    meta = loader.get_slide_data(session, int(slide_index), flatten=False)
    screenshot = Path(meta["screenshot_path"])
    if not screenshot.exists():
        warnings.warn(f"Screenshot unavailable: {screenshot}")
        return
    trial = raw.loc[raw["slide_index"].eq(int(slide_index))].iloc[0]
    boxes = target_word_boxes(trial.get("objects_bboxes"), trial[phrase_column])
    image = plt.imread(screenshot)
    height, width = image.shape[:2]
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(image)
    for bbox in boxes:
        try:
            cx, cy, w, h = (float(bbox[k]) for k in ("cx", "cy", "w", "h"))
        except Exception:
            continue
        ax.add_patch(Rectangle((width / 2 + cx - w / 2, height / 2 - cy - h / 2), w, h, fill=False, linewidth=2))
    f = fixations[fixations["slide_index"].eq(int(slide_index))] if not fixations.empty else pd.DataFrame()
    if not f.empty:
        ax.scatter(width / 2 + f["x_mean"], height / 2 - f["y_mean"], s=np.maximum(30, f["duration"] * 300), alpha=0.65)
    ax.set_title(f"Target AOI and fixations — slide {slide_index}")
    ax.axis("off")
    plt.show()


In [ ]:

target_metrics = target_fixation_metrics(raw, fixations, flat, "critical_phrase")
trial_metrics = (raw[["slide_index","item_id","topic","condition","answer","response","correct","uncertain","critical_phrase","text_word_count"]]
                 .merge(quality[["slide_index","gaze_samples","usable_gaze","observed_gaze_duration_s"]],on="slide_index")
                 .merge(fix_summary,on="slide_index").merge(scan_summary,on="slide_index").merge(target_metrics,on="slide_index"))
eye_cols=["fixation_count","total_fixation_duration","mean_fixation_duration","scanpath_transition_count","scanpath_distance"]
trial_metrics.loc[~trial_metrics["usable_gaze"],eye_cols]=np.nan
trial_metrics["verification_return"] = trial_metrics["target_revisits"].fillna(0).gt(0)
if (~trial_metrics["target_bbox_found"]).any(): warnings.warn("At least one critical phrase could not be matched to recorded word bboxes.")
display(trial_metrics)
if representative is not None: plot_target_aoi(loader,SESSION,raw,fixations,representative,"critical_phrase")


## 9. Search-efficiency and verification summaries

The summary tables follow the process model in the documentation:

- acquisition rate addresses H1;
- TTFF EARLY/LATE addresses H2;
- pre-target fixations address H3;
- target revisits and post-target fixation measures address H4;
- accuracy by acquisition/verification profile addresses H5.

Topic and text-length diagnostics are included because this demo intentionally trades experimental control for ecological variety. A placement difference that is driven by one unusually difficult item should be visible before it is generalized.


In [ ]:

condition_summary=(trial_metrics.groupby("condition")
 .agg(n_trials=("slide_index","size"),accuracy=("correct","mean"),uncertainty_rate=("uncertain","mean"),usable_gaze_rate=("usable_gaze","mean"),
      target_seen_rate=("target_seen","mean"),mean_ttff_target_s=("ttff_target_s","mean"),mean_fixations_before_target=("fixations_before_target","mean"),
      mean_target_dwell_s=("target_dwell_s","mean"),mean_target_revisits=("target_revisits","mean"),
      mean_post_target_fixations=("post_target_fixation_count","mean"),mean_post_target_fixation_duration_s=("post_target_fixation_duration_s","mean"),
      mean_fixation_count=("fixation_count","mean"),mean_scanpath_distance=("scanpath_distance","mean")).reset_index())
display(condition_summary)

outcome_summary=(trial_metrics.groupby(["correct","uncertain"],dropna=False)
 .agg(n_trials=("slide_index","size"),target_seen_rate=("target_seen","mean"),mean_ttff_target_s=("ttff_target_s","mean"),
      mean_target_revisits=("target_revisits","mean"),mean_post_target_fixations=("post_target_fixation_count","mean"),
      mean_target_dwell_s=("target_dwell_s","mean")).reset_index())
display(outcome_summary)

topic_diagnostics=trial_metrics[["item_id","topic","condition","text_word_count","response","correct","target_seen","ttff_target_s","target_revisits","post_target_fixation_count","observed_gaze_duration_s"]].sort_values(["correct","ttff_target_s"],ascending=[True,False],na_position="last")
display(topic_diagnostics)


## 10. Visual diagnostics

Visualizations are used to inspect distributions and item heterogeneity, not only condition means. In an applied search task, a few difficult topics can dominate averages.

Review trial-level plots together with target acquisition and correctness. A long trial with many fixations may reflect extended search, repeated verification, unfamiliar content, or recording noise; the surrounding metrics help distinguish these possibilities.


In [ ]:

fig,axes=plt.subplots(2,2,figsize=(12,9))
trial_metrics.boxplot(column="ttff_target_s",by="condition",ax=axes[0,0]); axes[0,0].set_title("Target TTFF by position")
trial_metrics.boxplot(column="post_target_fixation_count",by="condition",ax=axes[0,1]); axes[0,1].set_title("Post-target fixations")
seen=(trial_metrics.dropna(subset=["target_seen"]).groupby("target_seen")["correct"].mean())
seen.plot(kind="bar",ax=axes[1,0]); axes[1,0].set_ylim(0,1); axes[1,0].set_title("Accuracy by target acquisition")
for condition,g in trial_metrics.dropna(subset=["text_word_count","ttff_target_s"]).groupby("condition"):
    axes[1,1].scatter(g["text_word_count"],g["ttff_target_s"],label=condition)
axes[1,1].set(xlabel="Text word count",ylabel="TTFF (s)",title="Search latency vs text length")
handles, labels = axes[1,1].get_legend_handles_labels()
if handles:
    axes[1,1].legend()
plt.suptitle(""); plt.tight_layout(); plt.show()


## 11. Export derived tables

The notebook writes only derived tables and metadata. Raw experiment files remain immutable. This separation allows the analysis to be rerun with different fixation parameters without changing the original evidence.

For group studies, aggregate exported trial-level tables only after adding participant identifiers and retaining item identifiers so participant and item variability can be modeled separately.


In [ ]:

trial_metrics.to_csv(analysis_dir/"trial_metrics.csv",index=False)
condition_summary.to_csv(analysis_dir/"condition_summary.csv",index=False)
outcome_summary.to_csv(analysis_dir/"outcome_summary.csv",index=False)
topic_diagnostics.to_csv(analysis_dir/"topic_diagnostics.csv",index=False)
quality.to_csv(analysis_dir/"data_quality_trials.csv",index=False)
quality_overview.to_csv(analysis_dir/"data_quality_summary.csv",index=False)
fixations.to_csv(analysis_dir/"fixations.csv",index=False); scanpaths.to_csv(analysis_dir/"scanpaths.csv",index=False)
if not sensitivity.empty: sensitivity.to_csv(analysis_dir/"fixation_sensitivity.csv",index=False)
provenance.to_csv(analysis_dir/"provenance.csv",index=False)
save_json(analysis_dir/"analysis_parameters.json",{"language":LANGUAGE,"fixation":FIXATION_PARAMS,"sensitivity_thresholds":SENSITIVITY_DISPERSION_THRESHOLDS})
print(analysis_dir)


## 12. Scientific interpretation and limitations

Interpret the workflow as **orientation → search → target acquisition → verification → response**, consistent with the demo documentation. No single gaze metric represents the entire process.

- Missing gaze is unavailable evidence, not evidence that the target was ignored.
- A target fixation establishes overt visual access but not comprehension.
- `target_revisits`, post-target fixations, and post-target dwell are compatible with verification, but they do not uniquely measure confidence or understanding.
- Topic and wording vary across trials, so EARLY/LATE contrasts are descriptive unless item effects are explicitly modeled.
- Text length and critical-phrase location should be inspected when interpreting unusually long TTFF values.
- Confirmatory research should preregister AOI definitions, gaze-quality criteria, fixation parameters, exclusions, and participant/item-aware statistical models.

**Documentation link:** `docs/basic_examples/text_search_demo.md`.
